# Apache Hudi Core Conceptions (2) - File Layouts

## 1. Configuration

In [1]:
%%sh
# deploy hudi bundle jar
hdfs dfs -copyFromLocal -f /usr/lib/hudi/hudi-spark-bundle.jar /tmp/hudi-spark-bundle.jar
hdfs dfs -ls /tmp/hudi-spark-bundle.jar

-rw-r--r--   1 emr-notebook hdfsadmingroup   61421977 2023-03-09 08:43 /tmp/hudi-spark-bundle.jar


In [2]:
%%configure -f
{
    "conf" : {
        "spark.jars":"hdfs:///tmp/hudi-spark-bundle.jar",            
        "spark.serializer":"org.apache.spark.serializer.KryoSerializer",
        "spark.sql.extensions":"org.apache.spark.sql.hudi.HoodieSparkSessionExtension",
        "spark.sql.catalog.spark_catalog":"org.apache.spark.sql.hudi.catalog.HoodieCatalog"
    }
}

ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
44,application_1678096020253_0064,spark,idle,Link,Link,None,


In [3]:
%env S3_BUCKET=apache-hudi-core-conceptions

env: S3_BUCKET=apache-hudi-core-conceptions


In [4]:
%%sql
set S3_BUCKET=apache-hudi-core-conceptions

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
47,application_1678096020253_0067,spark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [5]:
%env WORKSPACE=/home/emr-notebook/apache-hudi-core-conceptions

env: WORKSPACE=/home/emr-notebook/apache-hudi-core-conceptions


In [6]:
%%sh
# make workspace
mkdir -p $WORKSPACE
# deploy hudi-stat.sh, a utility shell script to output hudi table status
wget https://raw.githubusercontent.com/bluishglc/apache-hudi-core-conceptions/master/hudi-stat.sh -O $WORKSPACE/hudi-stat.sh &>/dev/null
chmod a+x $WORKSPACE/hudi-stat.sh
ls $WORKSPACE/hudi-stat.sh

/home/emr-notebook/apache-hudi-core-conceptions/hudi-stat.sh


In [7]:
%%html
<style>
table {float:left}
</style>

## 2. Test Case 1 - COW File Layouts

### 2.1. Test Plan

Step No.|Action|Volume Per Partition |Storage
:--------:|:------|:------|:----------
1|Insert|96MB|+1 Small File
2|Insert|14.6MB|+1 Big File
3|Insert|3.7MB|+1 Small File
4|Insert|188.5MB|+1 Big File, +1 Small File

### 2.2. Key Settings

KEY|DEFAULT VALUE|SET VALUE
:---|:---|:---
hoodie.parquet.max.file.size|125829120 ( 120MB )|Default Value
hoodie.parquet.small.file.limit|104857600 ( 100MB )|Default Value
hoodie.copyonwrite.record.size.estimate|1024|175

### 2.3. Set Variables

In [8]:
%%sql
set TABLE_NAME=reviews_cow_file_layouts_1

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [9]:
%env TABLE_NAME=reviews_cow_file_layouts_1

env: TABLE_NAME=reviews_cow_file_layouts_1


### 2.4. Create Table

In [10]:
%%sh
aws s3 rm s3://${S3_BUCKET}/${TABLE_NAME} --recursive &>/dev/null
rm -rf ${WORKSPACE}/${TABLE_NAME}
sleep 5

In [11]:
%%sql
drop table if exists ${TABLE_NAME}

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [12]:
%%sql
create table if not exists ${TABLE_NAME}(
    review_id string, 
    star_rating int, 
    review_body string, 
    review_date date, 
    year long,
    timestamp long,
    parity int
)
using hudi
location 's3://${S3_BUCKET}/${TABLE_NAME}'
partitioned by (parity)
options ( 
    type = 'cow',  
    primaryKey = 'review_id', 
    preCombineField = 'timestamp',
    hoodie.copyonwrite.record.size.estimate = '175'
);

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

### 2.5. Insert 96MB ( +1 Small File )

In [13]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 2003;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [14]:
%%sh
${WORKSPACE}/hudi-stat.sh s3://${S3_BUCKET}/${TABLE_NAME} timeline commits storage


[ TIMELINE ]

╔═════╤═══════════════════╤════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │        │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309084413830 │ commit │ COMPLETED │ 03-09 08:44 │ 03-09 08:44 │ 03-09 08:45 ║
╚═════╧═══════════════════╧════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔═══════════════════╤═════════════════════╤═══════════════════╤═════════════════════╤══════════════════════════╤═══════════════════════╤══════════════════════════════╤══════════════╗
║ CommitTime        │ Total Bytes Written │ Total Files Added │ Total Files Updated │ Total Partitions Written │ Total Records Written │ Total Update Records Written │ Total Errors ║
╠═══════════════════╪═════════════════════╪════════════════

### 2.6. Insert 14.6MB ( +1 Big File )

In [15]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 1998;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [16]:
%%sh
${WORKSPACE}/hudi-stat.sh s3://${S3_BUCKET}/${TABLE_NAME} timeline commits storage


[ TIMELINE ]

╔═════╤═══════════════════╤════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │        │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309084413830 │ commit │ COMPLETED │ 03-09 08:44 │ 03-09 08:44 │ 03-09 08:45 ║
╟─────┼───────────────────┼────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309084529480 │ commit │ COMPLETED │ 03-09 08:45 │ 03-09 08:45 │ 03-09 08:46 ║
╚═════╧═══════════════════╧════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔═══════════════════╤═════════════════════╤═══════════════════╤═════════════════════╤══════════════════════════╤═══════════════════════╤══════════════════════════════╤══════════════╗
║ CommitTime        │ Total Bytes Written │ Total Files Adde

### 2.7. Insert 3.7MB ( +1 Small File )

In [17]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 1997;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [18]:
%%sh
${WORKSPACE}/hudi-stat.sh s3://${S3_BUCKET}/${TABLE_NAME} timeline commits storage


[ TIMELINE ]

╔═════╤═══════════════════╤════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │        │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309084413830 │ commit │ COMPLETED │ 03-09 08:44 │ 03-09 08:44 │ 03-09 08:45 ║
╟─────┼───────────────────┼────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309084529480 │ commit │ COMPLETED │ 03-09 08:45 │ 03-09 08:45 │ 03-09 08:46 ║
╟─────┼───────────────────┼────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 2   │ 20230309084631035 │ commit │ COMPLETED │ 03-09 08:46 │ 03-09 08:46 │ 03-09 08:46 ║
╚═════╧═══════════════════╧════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔═══════════════════╤═════════════════════╤══════════════════

### 2.8. Insert 188.5MB ( +1 Big File, +1 Small File )

In [19]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 2007;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [20]:
%%sh
${WORKSPACE}/hudi-stat.sh s3://${S3_BUCKET}/${TABLE_NAME} timeline commits storage


[ TIMELINE ]

╔═════╤═══════════════════╤════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │        │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309084413830 │ commit │ COMPLETED │ 03-09 08:44 │ 03-09 08:44 │ 03-09 08:45 ║
╟─────┼───────────────────┼────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309084529480 │ commit │ COMPLETED │ 03-09 08:45 │ 03-09 08:45 │ 03-09 08:46 ║
╟─────┼───────────────────┼────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 2   │ 20230309084631035 │ commit │ COMPLETED │ 03-09 08:46 │ 03-09 08:46 │ 03-09 08:46 ║
╟─────┼───────────────────┼────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 3   │ 20230309084705623 │ commit │ COMPLETED │ 03-09 08:47 │ 03-09 08:47 

## 3. Test Case 2 -  MOR File Layouts

### 3.1. Test Plan

Step No.|Action|Volume Per Partition |Storage
:--------:|:------|:------|:----------
1|Insert|41MB|+1 Base File
2|Update|7% of Total Records| +1 Log File
3|Update|5% of Total Records| +1 Log File

### 3.2. Key Settings

KEY|DEFAULT VALUE|SET VALUE
:---|:---|:---
hoodie.copyonwrite.record.size.estimate|1024|175

### 3.3. Set Variables

In [45]:
%%sql
set TABLE_NAME=reviews_mor_file_layouts_1

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [46]:
%env TABLE_NAME=reviews_mor_file_layouts_1

env: TABLE_NAME=reviews_mor_file_layouts_1


In [47]:
%%sql
set YEAR=1999

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [24]:
%%sql
select
    l.*, l.records_num / r.total as percentage
from
    (select star_rating, count(1) as records_num from reviews where year=${YEAR} group by star_rating) as l
join
    (select count(1) as total from reviews where year=${YEAR}) as r
order by 
    star_rating asc;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

### 3.4. Create Table

In [25]:
%%sh
aws s3 rm s3://${S3_BUCKET}/${TABLE_NAME} --recursive &>/dev/null
rm -rf ${WORKSPACE}/${TABLE_NAME}
sleep 5

In [26]:
%%sql
drop table if exists ${TABLE_NAME}

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [27]:
%%sql
create table if not exists ${TABLE_NAME}(
    review_id string, 
    star_rating int, 
    review_body string, 
    review_date date, 
    year long,
    timestamp long,
    parity int
)
using hudi
location 's3://${S3_BUCKET}/${TABLE_NAME}'
partitioned by (parity)
options ( 
    type = 'mor',  
    primaryKey = 'review_id', 
    preCombineField = 'timestamp',
    hoodie.copyonwrite.record.size.estimate = '175'
);

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

### 3.5. Insert 41MB ( +1 Base File )

In [28]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = ${YEAR}

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [29]:
%%sh
${WORKSPACE}/hudi-stat.sh s3://${S3_BUCKET}/${TABLE_NAME} timeline commits storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309084842147 │ deltacommit │ COMPLETED │ 03-09 08:48 │ 03-09 08:48 │ 03-09 08:49 ║
╚═════╧═══════════════════╧═════════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔═══════════════════╤═════════════════════╤═══════════════════╤═════════════════════╤══════════════════════════╤═══════════════════════╤══════════════════════════════╤══════════════╗
║ CommitTime        │ Total Bytes Written │ Total Files Added │ Total Files Updated │ Total Partitions Written │ Total Records Written │ Total Update Records Written │ Total Errors ║
╠═══════════════════╪════════

### 3.6. Update 7% Records ( +1 Log File )

In [30]:
%%sql
-- 1 star records take about 7% in total
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    star_rating = 1
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [31]:
%%sh
${WORKSPACE}/hudi-stat.sh s3://${S3_BUCKET}/${TABLE_NAME} timeline commits storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309084842147 │ deltacommit │ COMPLETED │ 03-09 08:48 │ 03-09 08:48 │ 03-09 08:49 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309084928795 │ deltacommit │ COMPLETED │ 03-09 08:49 │ 03-09 08:49 │ 03-09 08:49 ║
╚═════╧═══════════════════╧═════════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔═══════════════════╤═════════════════════╤═══════════════════╤═════════════════════╤══════════════════════════╤═══════════════════════╤══════════════════════════════╤══════════════╗
║ CommitTime        

### 3.7. Update 5% ( +1 Log File )

In [32]:
%%sql
-- 1 star records take about 5% in total
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    star_rating = 2
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [33]:
%%sh
${WORKSPACE}/hudi-stat.sh s3://${S3_BUCKET}/${TABLE_NAME} timeline commits storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309084842147 │ deltacommit │ COMPLETED │ 03-09 08:48 │ 03-09 08:48 │ 03-09 08:49 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309084928795 │ deltacommit │ COMPLETED │ 03-09 08:49 │ 03-09 08:49 │ 03-09 08:49 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 2   │ 20230309085003195 │ deltacommit │ COMPLETED │ 03-09 08:50 │ 03-09 08:50 │ 03-09 08:50 ║
╚═════╧═══════════════════╧═════════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔══════════

### 3.8. Insert 41MB ( +1 Base File )

In [48]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 2003

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [49]:
%%sh
${WORKSPACE}/hudi-stat.sh s3://${S3_BUCKET}/${TABLE_NAME} timeline commits storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309084842147 │ deltacommit │ COMPLETED │ 03-09 08:48 │ 03-09 08:48 │ 03-09 08:49 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309084928795 │ deltacommit │ COMPLETED │ 03-09 08:49 │ 03-09 08:49 │ 03-09 08:49 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 2   │ 20230309085003195 │ deltacommit │ COMPLETED │ 03-09 08:50 │ 03-09 08:50 │ 03-09 08:50 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 3   │ 20230309091407310

### 3.6. Update 7% Records ( +1 Log File )

In [50]:
%%sql
-- 1 star records take about 7% in total
update
    ${TABLE_NAME}
set             
    review_body = concat(uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid(),uuid()),
    timestamp = unix_timestamp(current_timestamp())
where
    year = 2003 and star_rating = 5
;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [51]:
%%sh
${WORKSPACE}/hudi-stat.sh s3://${S3_BUCKET}/${TABLE_NAME} timeline commits storage


[ TIMELINE ]

╔═════╤═══════════════════╤═════════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action      │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │             │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪═════════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309084842147 │ deltacommit │ COMPLETED │ 03-09 08:48 │ 03-09 08:48 │ 03-09 08:49 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309084928795 │ deltacommit │ COMPLETED │ 03-09 08:49 │ 03-09 08:49 │ 03-09 08:49 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 2   │ 20230309085003195 │ deltacommit │ COMPLETED │ 03-09 08:50 │ 03-09 08:50 │ 03-09 08:50 ║
╟─────┼───────────────────┼─────────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 3   │ 20230309091407310

## 4. Test Case 3 - COW Default Settings

### 4.1. Set Variables

In [34]:
%%sql
set TABLE_NAME=reviews_cow_file_layouts_2

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [35]:
%env TABLE_NAME=reviews_cow_file_layouts_2

env: TABLE_NAME=reviews_cow_file_layouts_2


## 4.2 Create Table

In [36]:
%%sh
echo $(basename s3://${S3_BUCKET}/${TABLE_NAME})
aws s3 rm s3://${S3_BUCKET}/${TABLE_NAME} --recursive &>/dev/null
rm -rf ${WORKSPACE}/${TABLE_NAME}
sleep 5

reviews_cow_file_layouts_2


In [37]:
%%sql
drop table if exists ${TABLE_NAME}

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [38]:
%%sql
create table if not exists ${TABLE_NAME}(
    review_id string, 
    star_rating int, 
    review_body string, 
    review_date date, 
    year long,
    timestamp long,
    parity int
)
using hudi
location 's3://${S3_BUCKET}/${TABLE_NAME}'
partitioned by (parity)
options ( 
    type = 'cow',  
    primaryKey = 'review_id', 
    preCombineField = 'timestamp'
    -- hoodie.copyonwrite.record.size.estimate = '175'
);

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

### 4.3. Batch 1 - Insert ( 0 -> 96MB / Partition )

In [39]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 2003;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [40]:
%%sh
${WORKSPACE}/hudi-stat.sh s3://${S3_BUCKET}/${TABLE_NAME} timeline commits storage


[ TIMELINE ]

╔═════╤═══════════════════╤════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │        │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309085048622 │ commit │ COMPLETED │ 03-09 08:50 │ 03-09 08:51 │ 03-09 08:51 ║
╚═════╧═══════════════════╧════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔═══════════════════╤═════════════════════╤═══════════════════╤═════════════════════╤══════════════════════════╤═══════════════════════╤══════════════════════════════╤══════════════╗
║ CommitTime        │ Total Bytes Written │ Total Files Added │ Total Files Updated │ Total Partitions Written │ Total Records Written │ Total Update Records Written │ Total Errors ║
╠═══════════════════╪═════════════════════╪════════════════

### 4.3. Batch 2 - Insert ( 96MB -> 417MB / Partition )

In [41]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 2010;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [42]:
%%sh
${WORKSPACE}/hudi-stat.sh s3://${S3_BUCKET}/${TABLE_NAME} timeline commits storage


[ TIMELINE ]

╔═════╤═══════════════════╤════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │        │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230309085048622 │ commit │ COMPLETED │ 03-09 08:50 │ 03-09 08:51 │ 03-09 08:51 ║
╟─────┼───────────────────┼────────┼───────────┼─────────────┼─────────────┼─────────────╢
║ 1   │ 20230309085135280 │ commit │ COMPLETED │ 03-09 08:51 │ 03-09 08:52 │ 03-09 08:53 ║
╚═════╧═══════════════════╧════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔═══════════════════╤═════════════════════╤═══════════════════╤═════════════════════╤══════════════════════════╤═══════════════════════╤══════════════════════════════╤══════════════╗
║ CommitTime        │ Total Bytes Written │ Total Files Adde